# GPU trial stability in Phases A and B

Compare repeated GPU measurements for the same framework and executed settings, then inspect nearby numeric settings. Phases A and B are analyzed separately because their models and workloads differ.

This notebook reads saved logs and SQLite trials; it launches no GPU benchmarks. Install the checkout with `pip install -e ".[notebook]"` first.

**What can be established:** identical-settings repeat ranges, sample standard deviation, coefficient of variation, and deviations from each group's median. Nearby settings also show parameter effects, so their differences do not establish GPU noise. Small repeat counts and incomplete environment metadata limit conclusions about stability.


In [ ]:
from pathlib import Path
import json

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/llm_optimizer").is_dir())
RUN_DIRS = sorted({
    p.parent for p in (ROOT / "experiment_results").glob("*/*-measured.jsonl")
})
# For one run only: RUN_DIRS = [ROOT / "experiment_results" / "optuna-phase-b-v1"]
# Add other saved run directories here; identical paths are read only once.
PHASES = ["A", "B"]
FRAMEWORKS = ["vllm", "sglang", "trtllm"]
MIN_REPEATS = 2             # Two points establish a range, not a reliable noise distribution.
DEVIATION_ALERT_PCT = 10.0  # Descriptive alert, not a statistical significance test.
MAX_NUMERIC_CHANGES = 1
INCLUDE_UNMATCHED = False   # Default: require a COMPLETE trial matched to a log record.
EXCLUDE_DUPLICATE_RECORDS = True  # Identical full JSON records may be copied artifacts.
TOP_N = 12
SAVE_REPORT = True
REPORT_DIR = ROOT / "experiment_results" / "phase_stability_analysis"


## Comparison rules

A comparison holds phase, framework, model, full client workload (including request count, seed, and concurrency), request fingerprints, and recorded environment metadata fixed. It can span samplers and objectives, since these do not change an already selected engine configuration.

Actual native server arguments define exact matches. JSON-valued flags are decoded. TRT-LLM YAML configs are resolved from the saved run and compared by contents, so moved file paths do not create false differences. Missing flags are not assumed equivalent to explicit flags, and graph capture arrays remain part of the configuration.

Below, similar means **at most one changed numeric setting** by default. Every other setting, including booleans, attention backends, scheduler policies, and graph capture settings, must match. Edit the absolute tolerances to suit the experiment.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from llm_optimizer.experiment.stability import (
    DEFAULT_TOLERANCES, METRICS, load_measurements, eligible_records,
    exact_repeat_summary, similar_parameter_pairs,
)

NUMERIC_TOLERANCES = dict(DEFAULT_TOLERANCES)
# Examples: NUMERIC_TOLERANCES["max_batch_size"] = 8
#           NUMERIC_TOLERANCES["free_gpu_memory_fraction"] = 0.05
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 20)
NUMERIC_TOLERANCES


In [ ]:
audit = load_measurements(RUN_DIRS)
selected, excluded = eligible_records(
    audit["records"], include_unmatched=INCLUDE_UNMATCHED,
    exclude_duplicate_records=EXCLUDE_DUPLICATE_RECORDS,
)
selected = [r for r in selected if r["phase"] in PHASES and r["framework"] in FRAMEWORKS]
exact = exact_repeat_summary(selected, min_repeats=MIN_REPEATS, alert_pct=DEVIATION_ALERT_PCT)
neighbors = similar_parameter_pairs(
    selected, tolerances=NUMERIC_TOLERANCES, max_changes=MAX_NUMERIC_CHANGES,
)

OBSERVATION_COLUMNS = [
    "observation_id", "phase", "framework", "concurrency", "study_name", "trial_number",
    "trial_state", "target", "sampler", "datetime_start", "datetime_complete",
    "context_id", "config_id", "exact_group_id", "source_file", "source_line",
    "measurement_valid", "server_config", "client_args", "identity_warnings", "exclusion_reason",
]
def observation_frame(rows):
    return pd.DataFrame([
        {**{k: r.get(k) for k in OBSERVATION_COLUMNS},
         **{k: r["metrics"].get(k) for k in (*METRICS, "completed", "total_requests", "failed")}}
        for r in rows
    ], columns=OBSERVATION_COLUMNS + list(METRICS) + ["completed", "total_requests", "failed"])

observations = observation_frame(selected)
excluded_observations = observation_frame(excluded)
trials = pd.DataFrame(audit["trials"], columns=[
    "run_dir", "phase", "framework", "concurrency", "study_name", "trial_number",
    "trial_state", "sampler", "target", "failure_reason", "datetime_start", "datetime_complete",
])
groups = pd.DataFrame(exact["groups"])
repeat_metrics = pd.DataFrame(exact["metrics"])
deviations = pd.DataFrame(exact["deviations"])
neighbor_pairs = pd.DataFrame(neighbors)

print(f"Loaded {len(audit['records'])} measurement records and {len(trials)} SQLite trials.")
print(f"Analyzing {len(observations)} valid matched measurements; {len(excluded)} excluded records.")
print("SQLite adds identity and failure coverage; it is not counted as a second measurement.")
if audit["issues"]:
    display(pd.DataFrame(audit["issues"]))
if not excluded_observations.empty:
    display(excluded_observations.groupby(["phase", "framework", "exclusion_reason"], dropna=False).size().rename("records").reset_index())


## Metadata and failure coverage

Placeholder framework/CUDA/driver/model revisions and missing physical GPU identifiers are reported below. Matching recorded metadata does not establish that temperature, power limits, background load, engine defaults, or software were actually unchanged.

Failed server starts consume search trials but produce no successful measurement. Invalid/incomplete benchmarks, failed trials, ambiguous matches, and unavailable native config files are excluded from repeat statistics. Full-record duplicates are excluded by default; independently measured trials with merely similar metric values are retained.


In [ ]:
warning_rows = [
    {"phase": r["phase"], "framework": r["framework"], "warning": warning,
     "observation_id": r["observation_id"]}
    for r in selected for warning in r["identity_warnings"]
]
identity_warnings = pd.DataFrame(warning_rows, columns=["phase", "framework", "warning", "observation_id"])
if not identity_warnings.empty:
    display(identity_warnings.groupby(["phase", "framework", "warning"]).size().rename("observations").reset_index())
if not trials.empty:
    display(trials.groupby(["phase", "framework", "trial_state"]).size().unstack(fill_value=0))
    failures = trials[trials["trial_state"] == "FAIL"]
    if not failures.empty:
        display(failures.groupby(["phase", "framework", "failure_reason"], dropna=False).size().rename("trials").reset_index())


## Reading the variation measures

For each exact group and metric:

- `min` / `max`: observed run-level range.
- `spread_pct`: `100 × (max − min) / median`.
- `cv_pct`: sample standard deviation divided by mean, as a percentage.
- `max_abs_deviation_pct`: largest absolute deviation from the group median.
- `n`: number of measurements, which should always accompany a variance claim.

The alert threshold applies to individual deviations from the median. It is a descriptive flag. For latency, these are variations **between runs' p95 latency values**, not the distribution of individual request latencies. No metric is normalized to a best trial, and these ranges are not confidence intervals.


In [ ]:
def phase_rows(frame, phase):
    if frame.empty:
        return frame
    return frame[(frame["phase"] == phase) & frame["framework"].isin(FRAMEWORKS)]

def show_phase(phase):
    obs = phase_rows(observations, phase)
    phase_groups = phase_rows(groups, phase)
    repeated = phase_rows(repeat_metrics, phase)
    nearby = phase_rows(neighbor_pairs, phase)
    coverage = []
    for framework in FRAMEWORKS:
        g = phase_groups[phase_groups["framework"] == framework] if not phase_groups.empty else phase_groups
        repeated_groups = g[g["n"] >= MIN_REPEATS] if not g.empty else g
        count = int((obs["framework"] == framework).sum())
        coverage.append({
            "framework": framework, "valid_measurements": count,
            "exact_configs": len(g), "repeated_configs": len(repeated_groups),
            "measurements_in_repeated_configs": int(repeated_groups["n"].sum()) if not repeated_groups.empty else 0,
            "singleton_configs": int((g["n"] == 1).sum()) if not g.empty else 0,
        })
    display(pd.DataFrame(coverage))
    t = phase_rows(trials, phase)
    if not t.empty:
        display(t.groupby(["framework", "trial_state"]).size().unstack(fill_value=0))
    if repeated.empty:
        print(f"Phase {phase}: no exact groups meet MIN_REPEATS={MIN_REPEATS}; stability is not established.")
    else:
        print("Range of observed deviations, by framework and metric:")
        population = phase_rows(deviations, phase)
        display(population.groupby(["framework", "metric"])["abs_deviation_pct"].agg(
            observations="count", median="median", p90=lambda s: s.quantile(0.90),
            p95=lambda s: s.quantile(0.95), maximum="max",
        ).reset_index())
        print("These percentiles describe saved observations; heavily repeated configs contribute more points.")
        for framework in FRAMEWORKS:
            rows = repeated[repeated["framework"] == framework]
            if rows.empty:
                print(f"{framework}: insufficient exact repeats.")
                continue
            print(f"{framework}: largest exact-settings variation (inspect repeat counts and metadata).")
            display(rows.sort_values("max_abs_deviation_pct", ascending=False)[[
                "exact_group_id", "concurrency", "metric", "n", "min", "max", "median",
                "spread_pct", "cv_pct", "max_abs_deviation_pct", "alert", "studies", "server_config",
            ]].head(TOP_N))
        fig, axes = plt.subplots(1, len(METRICS), figsize=(16, 4))
        for ax, metric in zip(axes, METRICS):
            for framework in FRAMEWORKS:
                values = population[(population["framework"] == framework) & (population["metric"] == metric)]["deviation_pct"].dropna()
                if len(values):
                    ax.hist(values, bins=20, histtype="step", label=f"{framework} (n={len(values)})")
            ax.axvline(-DEVIATION_ALERT_PCT, color="gray", linestyle="--")
            ax.axvline(DEVIATION_ALERT_PCT, color="gray", linestyle="--")
            ax.set(title=metric, xlabel="Deviation from exact-group median (%)", ylabel="Measurements")
            if ax.get_legend_handles_labels()[0]:
                ax.legend()
        fig.suptitle(f"Phase {phase}: identical-settings deviations")
        fig.tight_layout()
        if SAVE_REPORT:
            REPORT_DIR.mkdir(parents=True, exist_ok=True)
            fig.savefig(REPORT_DIR / f"phase-{phase}-exact-deviations.png", dpi=180, bbox_inches="tight")
        plt.show()
    print("Nearby numeric settings: median differences mix parameter effects and measurement variation.")
    if nearby.empty:
        print("No pairs meet the selected numeric tolerances with other settings held fixed.")
    else:
        for framework in FRAMEWORKS:
            rows = nearby[nearby["framework"] == framework]
            if rows.empty:
                continue
            print(framework)
            display(rows.sort_values("median_gap_pct", ascending=False)[[
                "metric", "concurrency", "left_group_id", "right_group_id", "left_n", "right_n",
                "left_median", "right_median", "left_min", "left_max", "right_min", "right_max",
                "median_gap_pct", "parameter_changes",
            ]].head(TOP_N))


## Phase A

Exact repetitions and nearby-setting comparisons for the Phase A model and workloads only. Workload or environment differences create separate contexts even within this phase.


In [ ]:
show_phase("A")


## Phase B

Exact repetitions and nearby-setting comparisons for the Phase B model and workloads only. Results are not pooled with Phase A.


In [ ]:
show_phase("B")


## Inspect an exact configuration

Select a group ID from either phase's tables. The default is the group with the largest observed absolute deviation. Inspect source lines, study/trial identity, settings, and all raw values before attributing a large difference to GPU variation. The timestamp comes from the matched SQLite trial; it includes server startup and benchmark execution.


In [ ]:
GROUP_ID = None  # Paste a full exact_group_id to choose another configuration.
if GROUP_ID is None and not repeat_metrics.empty:
    GROUP_ID = repeat_metrics.sort_values("max_abs_deviation_pct", ascending=False).iloc[0]["exact_group_id"]

if GROUP_ID:
    details = observations[observations["exact_group_id"] == GROUP_ID].sort_values(
        ["datetime_start", "study_name", "trial_number"], na_position="last"
    )
    if details.empty:
        print("No matching group in the selected observations.")
    else:
        print("Executed server settings:")
        display(details.iloc[0]["server_config"])
        print("Identity limitations:")
        display(details.iloc[0]["identity_warnings"])
        display(details[["phase", "framework", "concurrency", "study_name", "trial_number", "sampler",
                         "datetime_start", *METRICS, "source_file", "source_line"]])
        fig, axes = plt.subplots(1, len(METRICS), figsize=(16, 4))
        for ax, metric in zip(axes, METRICS):
            values = details[metric].to_numpy()
            center = np.median(values)
            ax.plot(range(len(values)), values, marker="o")
            ax.axhline(center, color="gray", label="Group median")
            ax.set(title=metric, xlabel="Observation order (not elapsed time)", ylabel="tokens/s" if metric == "output_throughput" else "ms")
            ax.legend()
        fig.suptitle(f"Identical settings: {details.iloc[0]['phase']} / {details.iloc[0]['framework']} / n={len(details)}")
        fig.tight_layout()
        if SAVE_REPORT:
            REPORT_DIR.mkdir(parents=True, exist_ok=True)
            fig.savefig(REPORT_DIR / "selected-exact-group.png", dpi=180, bbox_inches="tight")
        plt.show()
else:
    print("No repeated configurations available for drill-down.")


## Save the analysis

Exports keep the raw observation references, exclusions, trial failures, identity warnings, exact ranges, and similar-parameter differences. Settings and comparison definitions are saved alongside the tables. Only the analysis directory is written; the source logs and studies stay unchanged.

These are historical checks, not independent reruns. A large exact-group range warrants controlled repeats with a pinned environment. Sparse groups or stable-looking pairs do not establish general GPU stability.


In [ ]:
if SAVE_REPORT:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    tables = {
        "observations": observations, "excluded_observations": excluded_observations,
        "trial_states": trials, "exact_groups": groups, "exact_metrics": repeat_metrics,
        "exact_deviations": deviations, "similar_pairs": neighbor_pairs,
        "identity_warnings": identity_warnings, "source_issues": pd.DataFrame(audit["issues"]),
    }
    for name, frame in tables.items():
        saved = frame.copy()
        for column in saved.select_dtypes(include="object").columns:
            saved[column] = saved[column].map(
                lambda value: json.dumps(value, sort_keys=True) if isinstance(value, (dict, list)) else value
            )
        saved.to_csv(REPORT_DIR / f"{name}.csv", index=False)
    manifest = {
        "run_dirs": [str(p.resolve()) for p in RUN_DIRS], "phases": PHASES, "frameworks": FRAMEWORKS,
        "min_repeats": MIN_REPEATS, "deviation_alert_pct": DEVIATION_ALERT_PCT,
        "numeric_tolerances": NUMERIC_TOLERANCES, "max_numeric_changes": MAX_NUMERIC_CHANGES,
        "include_unmatched": INCLUDE_UNMATCHED, "exclude_duplicate_records": EXCLUDE_DUPLICATE_RECORDS,
        "loaded_records": len(audit["records"]), "selected_records": len(selected), "excluded_records": len(excluded),
        "scope": "Historical GPU observations grouped by executed settings, workload fingerprints, and declared environment; neighboring settings include parameter effects.",
    }
    (REPORT_DIR / "analysis_settings.json").write_text(json.dumps(manifest, indent=2) + "\n")
    print(f"Saved analysis to {REPORT_DIR}")
else:
    print("Report export disabled.")
